# 第十四章 自动化深度研究智能体

在第十三章的旅行助手项目中，我们体验了如何将 HelloAgents 应用于一个多智能体产品。本章我们继续向前，聚焦「知识密集型应用」：**构建一个能够自动化执行深度研究任务的智能体助手。**

相比旅行规划，深度研究的难点在于信息的不断发散、事实的快速更新以及用户对引用来源的高要求。为了交付可信的研究报告，我们需要让智能体具备三个核心能力：

**（1）问题剖析**：将用户的开放主题拆解为可检索的查询语句。

**（2）多轮信息采集**：结合不同搜索 API 持续挖掘资料，并去重整合。

**（3）反思与总结**：依据阶段结果识别知识空白，决定是否继续检索，并生成结构化总结。

## 14.1 项目概述与架构设计

### 14.1.1 为什么需要深度研究助手

在信息爆炸的时代，传统的研究方式有几个痛点：

- **信息过载**：搜索引擎返回成千上万的结果，需要逐个点开阅读，才能找到有用的信息。
- **缺少结构**：即使找到了相关信息，也往往是碎片化的，缺少系统性的组织。
- **重复劳动**：每次研究新主题时，都需要重复
的过程。

**深度研究助手的核心价值：**

1. **节省时间**：将 1-2 小时的研究工作压缩到 5-10 分钟
2. **提高质量**：系统化的研究流程，避免遗漏重要信息
3. **可追溯**：记录所有搜索结果和来源，方便验证和引用
4. **可扩展**：可以轻松添加新的搜索引擎、数据源和分析工具

### 14.1.2 技术架构概览

系统采用前后端分离架构，分为四层（图 14.1）：

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/14-figures/14-1.png" alt="" width="85%"/>
  <p>图 14.1 深度研究助手技术架构</p>
</div>

- **前端层 (Vue3+TypeScript)**：全屏模态对话框 UI、Markdown 结果可视化
- **后端层 (FastAPI)**：API 路由（`/research/stream`）
- **智能体层 (HelloAgents)**：三个专门 Agent（TODO Planner、Task Summarizer、Report Writer）+ 两个核心工具（SearchTool、NoteTool）
- **外部服务层**：搜索引擎 + LLM 提供商

完整的研究请求数据流转过程（图 14.2）：

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/14-figures/14-2.png" alt="" width="85%"/>
  <p>图 14.2 深度研究助手数据流转过程</p>
</div>

1. 用户在前端输入研究主题
2. 前端通过 SSE 连接到 `/research/stream`
3. FastAPI 接收请求，创建研究状态
4. 调用研究规划 Agent，分解为 3 个子任务
5. 逐个执行每个子任务（搜索 → 总结 → 记录）
6. 调用报告生成 Agent，整合所有总结
7. 通过 SSE 推送进度和结果到前端
8. 前端实时更新任务状态、进度条、日志、报告

In [ ]:
# 项目目录结构
project_structure = """
helloagents-deepresearch/
├── backend/                    # 后端代码
│   ├── src/
│   │   ├── agent.py           # 核心协调器
│   │   ├── main.py            # FastAPI入口
│   │   ├── models.py          # 数据模型
│   │   ├── prompts.py         # Prompt模板
│   │   ├── config.py          # 配置管理
│   │   └── services/          # 服务层
│   │       ├── planner.py     # 规划服务
│   │       ├── summarizer.py  # 总结服务
│   │       ├── reporter.py    # 报告服务
│   │       └── search.py      # 搜索服务
│   ├── .env                   # 环境变量
│   ├── pyproject.toml         # 依赖管理
│   └── workspace/             # 研究笔记
│
└── frontend/                   # 前端代码
    ├── src/
    │   ├── App.vue            # 主组件
    │   ├── components/        # UI组件
    │   │   └── ResearchModal.vue
    │   └── composables/       # 组合式函数
    │       └── useResearch.ts
    ├── package.json           # npm依赖
    └── vite.config.ts         # 构建配置
"""
print(project_structure)

### 14.1.3 快速体验：5 分钟运行项目

In [ ]:
# 启动说明（在终端中执行，不可直接在此运行）

startup_backend = """
# 1. 进入后端目录
cd helloagents-deepresearch/backend

# 2. 安装依赖
# 方式1：使用 uv（推荐，更快的 Python 包管理器）
uv sync
# 方式2：使用 pip
pip install -e .

# 3. 配置环境变量
cp .env.example .env
# 编辑 .env，至少配置：
# LLM_PROVIDER=openai|deepseek|qwen
# LLM_API_KEY=<your key>
# SEARCH_API=duckduckgo|tavily

# 4. 启动后端
python src/main.py
# 成功后访问 http://localhost:8000
"""

startup_frontend = """
# 在新终端窗口执行
cd helloagents-deepresearch/frontend
npm install
npm run dev
# 成功后访问 http://localhost:5174
"""

print("后端启动命令:", startup_backend)
print("前端启动命令:", startup_frontend)

打开浏览器访问 http://localhost:5174，输入研究主题（如 `Datawhale是一个什么样的组织？`），点击
按钮。

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/14-figures/14-3.png" alt="" width="85%"/>
  <p>图 14.3 深度研究助手搜索页面</p>
</div>

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/14-figures/14-4.png" alt="" width="85%"/>
  <p>图 14.4 深度研究助手展开研究</p>
</div>

研究完成后，你会看到任务列表（子任务状态）、进度日志（所有操作记录）以及结构化的 Markdown 最终报告（包含来源引用）。

## 14.2 TODO 驱动的研究范式

### 14.2.1 什么是 TODO 驱动的研究

TODO 驱动的研究范式将复杂的研究主题分解为多个子任务，逐个执行并整合结果。核心思想是：**将
这个复杂任务转化为
的流程**。

**传统搜索方式 vs. TODO 驱动方式对比：**

```
传统方式：
  用户输入 → 搜索 → 返回10-20个链接 → 手动阅读整理 → 碎片化结果

TODO驱动方式：
  用户输入：Datawhale是一个什么样的组织？
  
  系统规划：
    ├─ TODO 1：Datawhale的基本信息（组织定位）
    ├─ TODO 2：Datawhale的主要项目（核心内容）
    ├─ TODO 3：Datawhale的社区文化（价值观）
    └─ TODO 4：Datawhale的影响力（社会贡献）
  
  系统执行：对每个TODO → 搜索 → 总结 → 记录来源
  
  系统整合：生成结构化报告（各部分 + 参考文献）
```

### 14.2.2 三阶段研究流程

TODO 驱动的研究流程分为三个阶段（图 14.5）：

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/14-figures/14-5.png" alt="" width="85%"/>
  <p>图 14.5 TODO 驱动的研究流程</p>
</div>

In [ ]:
# 阶段1：规划 - 规划 Agent 输出的子任务 JSON 示例
import json

example_todo_list = [
    {
        "title": "Datawhale的基本信息",
        "intent": "了解Datawhale的组织定位、成立时间、发展历程",
        "query": "Datawhale organization introduction history 2024"
    },
    {
        "title": "Datawhale的主要项目",
        "intent": "了解Datawhale的核心开源项目和教程",
        "query": "Datawhale projects tutorials open source 2024"
    },
    {
        "title": "Datawhale的社区文化",
        "intent": "了解Datawhale的价值观和社区运营方式",
        "query": "Datawhale community culture values learner 2024"
    }
]

print("规划阶段输出 - 子任务列表:")
print(json.dumps(example_todo_list, ensure_ascii=False, indent=2)) # 美化输出，保持中文显示，不转义为unicode，并且缩进2个空格

In [ ]:
# 阶段2：执行 - 搜索结果示例
example_search_results = {
    "results": [
        {
            "title": "What is Datawhale?",
            "url": "https://github.com/datawhalechina",
            "snippet": "Datawhale is an open-source organization focused on data science and AI..."
        },
        {
            "title": "Datawhale 关于我们",
            "url": "https://datawhale.club/about",
            "snippet": "Datawhale，一个专注于AI领域的学习社区，成立于2018年..."
        }
    ]
}

# 阶段2执行流程伪代码
execution_flow = """
for task in todo_list:
    # 1. 搜索资料
    search_results = search_tool.run({
        "input": task.query,
        "backend": "tavily",
        "mode": "structured",
        "max_results": 5
    })

    # 2. 总结搜索结果
    summary = summarizer_agent.run(
        task=task,
        search_results=search_results
    )

    # 3. 记录总结和来源
    note_tool.run({
        "action": "create",
        "title": task.title,
        "content": f"## {task.title}\n\n{summary}\n\n## 来源\n{sources}",
        "tags": ["research", "summary"]
    })
"""

print("执行阶段流程:")
print(execution_flow)
print("\n搜索结果示例:")
print(json.dumps(example_search_results, ensure_ascii=False, indent=2))

In [ ]:
# 阶段2：执行 - 任务总结 Agent 输出示例
example_task_summary = """## Datawhale的基本信息

Datawhale是一个专注于数据科学与AI领域的开源组织，成立于2018年[1]。
组织的核心使命是
，致力于构建一个纯粹的学习社区[2]。

**核心定位：**

1. **开源教育平台**：提供高质量的AI和数据科学学习资源[1]
2. **学习者社区**：汇聚了数万名AI学习者和实践者[3]
3. **知识共享**：倡导开源精神，所有内容完全免费开放[2]

**发展历程：**

- **2018年**：Datawhale成立，发布首个开源教程[1]
- **2020年**：成为国内领先的AI学习社区之一[3]
- **2024年**：累计发布50+开源项目，影响10万+学习者[4]

## 来源

[1] https://github.com/datawhalechina
[2] https://datawhale.club/about
[3] https://www.zhihu.com/org/datawhale
[4] https://datawhale.cn
"""

print("任务总结示例（Markdown格式）:")
print(example_task_summary)

## 14.3 智能体系统设计

### 14.3.1 Agent 职责划分

三个专门的 Agent 各司其职（表 14.1）：

<div align="center">
  <p>表 14.1 三个 Agent 的职责划分</p>
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/14-figures/14-table-1.png" alt="" width="85%"/>
</div>

- **TODO Planner（研究规划专家）**：将研究主题分解为 3-5 个子任务，输出 JSON 格式的子任务列表
- **Task Summarizer（任务总结专家）**：总结搜索结果，提取关键信息，添加来源引用
- **Report Writer（报告撰写专家）**：整合所有子任务总结，生成最终 Markdown 格式报告

In [ ]:
# Agent 1：研究规划专家 - Prompt 设计
todo_planner_instructions = """
你是一个研究规划专家。你的任务是将用户的研究主题分解为3-5个子任务。

当前日期：{current_date}

研究主题：{research_topic}

请分析这个研究主题，将其分解为3-5个子任务。每个子任务应该：
1. 涵盖主题的一个重要方面
2. 有明确的研究目标
3. 可以通过搜索引擎找到相关资料

请以JSON格式返回子任务列表，每个子任务包含：
- title：任务标题（简洁明了）
- intent：任务意图（为什么要研究这个）
- query：搜索查询（用于搜索引擎的查询字符串，可以使用英文以获得更好的搜索结果）

示例输出：
[
  {{
    "title": "什么是多模态模型",
    "intent": "了解多模态模型的基础概念，为后续研究打下基础",
    "query": "multimodal model definition concept 2024"
  }},
  ...
]

请确保：
1. 子任务数量在3-5个之间
2. 子任务之间有逻辑关系（如从基础到应用，从现状到趋势）
3. 搜索查询能够准确找到相关资料
4. 只返回JSON，不要包含其他文本
"""

print("TODO Planner Prompt 模板（关键设计点）:")
print("  1. 包含当前日期 → 获取最新信息")
print("  2. 明确 JSON 格式 → 便于解析")
print("  3. 提供示例 → 帮助 LLM 理解期望输出")
print("  4. 强调约束 → 确保质量")
print(f"\nPrompt 长度: {len(todo_planner_instructions)} 字符")

In [ ]:
# Agent 2：任务总结专家 - Prompt 设计
task_summarizer_instructions = """
你是一个任务总结专家。你的任务是总结搜索结果，提取关键信息。

任务标题：{task_title}
任务意图：{task_intent}
搜索查询：{task_query}

搜索结果：
{search_results}

请仔细阅读以上搜索结果，提取关键信息，并以Markdown格式返回总结。

总结应该包含：
1. **核心观点**：搜索结果中的核心观点和结论
2. **关键数据**：重要的数字、日期、名称等
3. **来源引用**：为每个观点添加来源引用（使用[1]、[2]等标记）

请确保：
1. 总结简洁明了，避免冗余
2. 保留重要的细节和数据
3. 为每个观点添加来源引用
4. 使用Markdown格式（标题、列表、加粗等）
"""

# Agent 3：报告撰写专家 - Prompt 设计
report_writer_instructions = """
你是一个报告撰写专家。你的任务是整合所有子任务的总结，生成一份结构化的研究报告。

研究主题：{research_topic}

子任务总结：
{task_summaries}

请整合以上所有子任务的总结，生成一份结构化的研究报告。

报告应该包含：
1. **标题**：研究主题
2. **概述**：简要介绍研究主题和报告结构（2-3段）
3. **各个子任务的详细分析**：按照逻辑顺序组织（使用二级标题）
4. **总结**：总结研究的主要发现（1-2段）
5. **参考文献**：所有来源引用（按照子任务分组）

请确保：
1. 报告结构清晰，逻辑连贯
2. 消除重复的信息
3. 保留所有来源引用
4. 使用Markdown格式
"""

print("三个 Agent 的 Prompt 设计要点:")
print(f"  TODO Planner:    {len(todo_planner_instructions)} 字符 - 强调 JSON 输出")
print(f"  Task Summarizer: {len(task_summarizer_instructions)} 字符 - 强调来源引用")
print(f"  Report Writer:   {len(report_writer_instructions)} 字符 - 强调结构化输出")

### 14.3.2 ToolAwareSimpleAgent 的设计

在深度研究助手中，我们需要一个能够**记录工具调用**的 Agent，用于调试、日志记录、分析和实时进度展示。

`ToolAwareSimpleAgent` 在 `SimpleAgent` 的基础上增加了 `tool_call_listener` 回调参数，每次工具调用时都会触发该回调。

In [ ]:
# ToolAwareSimpleAgent 实现（需要在后端项目中运行）
tool_aware_agent_code = '''
from hello_agents import SimpleAgent, HelloAgentsLLM
from hello_agents.tools import ToolRegistry
from typing import Optional, Callable

class ToolAwareSimpleAgent(SimpleAgent):
    """继承 SimpleAgent，增加工具调用监听能力"""

    def __init__(
        self,
        name: str,
        system_prompt: str,
        llm: HelloAgentsLLM,
        tool_registry: Optional[ToolRegistry] = None,
        tool_call_listener: Optional[Callable] = None,
    ):
        super().__init__(
            name=name,
            system_prompt=system_prompt,
            llm=llm,
            tool_registry=tool_registry,
        )
        self._tool_call_listener = tool_call_listener

    def _execute_tool_call(self, tool_name: str, parameters: str) -> str:
        """执行工具调用，并通知监听器"""
        # 解析参数
        parsed_parameters = self._parse_parameters(parameters)

        # 调用工具（父类方法）
        result = super()._execute_tool_call(tool_name, parameters)

        # 通知监听器
        if self._tool_call_listener:
            self._tool_call_listener({
                "agent_name": self.name,
                "tool_name": tool_name,
                "parsed_parameters": parsed_parameters,
                "result": result,
            })

        return result
'''

# 使用示例
usage_example = '''
# 创建工具调用监听器
def tool_listener(call_info):
    print(f"Agent: {call_info['agent_name']}")
    print(f"工具: {call_info['tool_name']}")
    print(f"参数: {call_info['parsed_parameters']}")
    print(f"结果: {call_info['result']}")

# 创建 Agent（带监听器）
agent = ToolAwareSimpleAgent(
    name="研究助手",
    system_prompt="你是一个研究助手",
    llm=llm,
    tool_call_listener=tool_listener
)
'''

print("ToolAwareSimpleAgent 实现:")
print(tool_aware_agent_code)
print("使用示例:")
print(usage_example)

### 14.3.3 Agent 协作模式

三个 Agent 之间是**顺序协作**的关系（图 14.6）：

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/14-figures/14-6.png" alt="" width="85%"/>
  <p>图 14.6 Agent 协作流程</p>
</div>

顺序协作模式特点：线性流程、明确的输入输出、同一时间只有一个 Agent 工作。

In [ ]:
# DeepResearchAgent 核心协调器（伪代码，展示协作流程）
class DeepResearchAgent:
    """深度研究智能体 - 协调三个专门 Agent"""

    def __init__(self):
        # 实际项目中需要传入 llm 和 config
        # self.llm = HelloAgentsLLM(...)
        # 创建共享的工具调用监听器，将事件推送给前端
        # def tool_listener(call_info):
        #     self._emit_event({"type": "tool_call", ...})
        # self.planner = PlanningService(self.llm, tool_listener)
        # self.summarizer = SummarizationService(self.llm, tool_listener)
        # self.reporter = ReportingService(self.llm, tool_listener)
        pass

    def run(self, research_topic: str) -> str:
        """顺序执行三个阶段"""

        # 1. 规划阶段
        self._emit_event({"type": "status", "message": "正在规划研究任务..."})
        print(f"[规划阶段] 分解主题: {research_topic}")
        # todo_list = self.planner.plan_todo_list(research_topic)
        todo_list = ["任务1", "任务2", "任务3"]  # 模拟
        self._emit_event({"type": "tasks", "tasks": todo_list})

        # 2. 执行阶段
        task_summaries = []
        for task in todo_list:
            self._emit_event({"type": "status", "message": f"正在研究：{task}"})
            print(f"[执行阶段] 搜索并总结: {task}")
            # search_results = self.search_service.search(task.query)
            # summary = self.summarizer.summarize_task(task, search_results)
            # task_summaries.append((task, summary))
            self._emit_event({"type": "task_completed", "task": task})

        # 3. 报告阶段
        self._emit_event({"type": "status", "message": "正在生成报告..."})
        print("[报告阶段] 整合所有总结，生成最终报告")
        # report = self.reporter.generate_report(research_topic, task_summaries)
        # self._emit_event({"type": "report", "content": report})

        return "[报告生成完成]"

    def _emit_event(self, event: dict):
        """推送事件到前端（通过SSE）"""
        print(f"  → 事件: {event}")


# 演示协作流程
agent = DeepResearchAgent()
result = agent.run("Datawhale是一个什么样的组织？")

## 14.4 工具系统集成

### 14.4.1 SearchTool 扩展

在第七章 SearchTool 基础上，新增了 DuckDuckGo、Perplexity、SearXNG 等搜索引擎，并实现了 Advanced 模式（组合多个搜索引擎）。

<div align="center">
  <p>表 14.2 多搜索引擎对比</p>
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/14-figures/14-table-2.png" alt="" width="85%"/>
</div>

In [ ]:
# 搜索引擎配置（config.py）
from enum import Enum
from pydantic import BaseModel

class SearchAPI(str, Enum):
    TAVILY = "tavily"
    DUCKDUCKGO = "duckduckgo"
    PERPLEXITY = "perplexity"
    SEARXNG = "searxng"
    ADVANCED = "advanced"  # 组合多个搜索引擎

class Configuration(BaseModel):
    search_api: SearchAPI = SearchAPI.DUCKDUCKGO
    max_results: int = 5
    max_tokens_per_source: int = 2000

print("支持的搜索引擎:", [e.value for e in SearchAPI])
print("\n通过 .env 配置：SEARCH_API=tavily  # 无需修改代码")

In [ ]:
# 搜索结果处理工具函数
from typing import List, Set

def deduplicate_sources(sources: List[dict]) -> List[dict]:
    """去除重复的 URL"""
    seen_urls: Set[str] = set()
    unique_sources = []

    for source in sources:
        url = source.get("url", "")
        if url and url not in seen_urls:
            seen_urls.add(url)
            unique_sources.append(source)

    return unique_sources


def limit_source_tokens(source: dict, max_tokens: int = 2000) -> dict:
    """限制每个来源的 Token 数量（1 Token ≈ 4 字符）"""
    snippet = source.get("snippet", "")
    max_chars = max_tokens * 4

    if len(snippet) > max_chars:
        snippet = snippet[:max_chars] + "..."

    return {**source, "snippet": snippet}


# 测试
test_sources = [
    {"url": "https://example.com/a", "snippet": "内容A" * 100},
    {"url": "https://example.com/b", "snippet": "内容B"},
    {"url": "https://example.com/a", "snippet": "重复URL"},  # 重复
]

deduped = deduplicate_sources(test_sources)
print(f"去重前: {len(test_sources)} 条 → 去重后: {len(deduped)} 条")

limited = [limit_source_tokens(s, max_tokens=10) for s in deduped]
for s in limited:
    print(f"  URL: {s['url']}  snippet 长度: {len(s['snippet'])} 字符")

### 14.4.2 NoteTool 使用

`NoteTool` 是第九章集成的内置工具，用于持久化研究进度。研究笔记存储在指定工作空间目录中，每个笔记是一个 Markdown 文件。

```
workspace/
├── notes/
│   ├── 1.md   # 任务1的笔记
│   ├── 2.md   # 任务2的笔记
│   └── ...
└── reports/
    └── final_report.md
```

In [ ]:
# NotesService - 封装 NoteTool 的使用
notes_service_code = '''
class NotesService:
    def __init__(self, workspace: str):
        # from hello_agents.tools import NoteTool
        self.note_tool = NoteTool(workspace=workspace)

    def save_task_summary(
        self,
        task: TodoItem,
        search_results: List[dict],
        summary: str
    ):
        """保存任务总结到笔记"""
        content = self._format_note_content(task, search_results, summary)

        self.note_tool.run({
            "action": "create",
            "title": f"任务{task.id}：{task.title}",
            "content": content,
            "tags": ["research", "summary"]
        })

    def _format_note_content(
        self, task, search_results, summary
    ) -> str:
        content = f"# 任务{task.id}：{task.title}\n\n"
        content += f"## 任务信息\n\n"
        content += f"- **意图**：{task.intent}\n"
        content += f"- **查询**：{task.query}\n\n"

        content += f"## 搜索结果\n\n"
        for idx, result in enumerate(search_results, start=1):
            content += f"[{idx}] {result['title']}\n"
            content += f"URL: {result['url']}\n"
            content += f"摘要: {result['snippet']}\n\n"

        content += f"## 总结\n\n{summary}\n"
        return content
'''

print("NotesService 代码:")
print(notes_service_code)

### 14.4.3 ToolRegistry 工具管理

`ToolRegistry` 是 HelloAgents 框架的工具注册表（第七章实现），统一管理所有工具的注册和调用。工具调用流程如图 14.7 所示：

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/14-figures/14-7.png" alt="" width="85%"/>
  <p>图 14.7 工具调用流程</p>
</div>

In [ ]:
# ToolRegistry 使用示例（后端项目中运行）
tool_registry_code = '''
from hello_agents import ToolAwareSimpleAgent
from hello_agents.tools import ToolRegistry, SearchTool, NoteTool

# 1. 创建工具
search_tool = SearchTool(backend="hybrid")
note_tool = NoteTool(workspace="./workspace/notes")

# 2. 创建注册表并注册工具
registry = ToolRegistry()
registry.register_tool(search_tool)
registry.register_tool(note_tool)

# 3. 创建使用工具注册表的 Agent
agent = ToolAwareSimpleAgent(
    name="研究助手",
    system_prompt="你是一个研究助手",
    llm=llm,
    tool_registry=registry
)

# Agent 生成工具调用指令（由 LLM 自动决定）：
# [TOOL_CALL:search_tool:{"input": "Datawhale组织", "backend": "tavily"}]
# ToolRegistry 解析 → 查找工具 → 调用 run() → 返回结果给 Agent
'''

print("ToolRegistry 使用示例:")
print(tool_registry_code)

## 14.5 服务层实现

四个核心服务是连接 Agent 和工具的桥梁，负责具体的业务逻辑。

### 14.5.1 任务规划服务（PlanningService）

核心职责：构建规划 Prompt → 调用 TODO Planner Agent → 解析 JSON 响应 → 验证子任务格式

In [ ]:
import re
import json
from typing import List, Optional
from datetime import datetime
from dataclasses import dataclass


@dataclass #利用装饰器自动生成 __init__、__repr__ 等方法
class TodoItem:
    id: int
    title: str
    intent: str
    query: str
#等价写法
class TodoItem:
    def __init__(self, id: int, title: str, intent: str, query: str):
        self.id = id
        self.title = title
        self.intent = intent
        self.query = query


class PlanningService:
    """任务规划服务"""

    def __init__(self, llm=None, tool_call_listener=None):
        self._llm = llm
        # 实际项目中：
        # self._agent = ToolAwareSimpleAgent(
        #     name="TODO Planner",
        #     system_prompt="你是一个研究规划专家",
        #     llm=llm,
        #     tool_call_listener=tool_call_listener
        # )

    def _get_current_date(self) -> str:
        return datetime.now().strftime("%Y年%m月%d日")

    def _extract_tasks(self, response: str) -> List[dict]:
        """从 Agent 响应中稳健地提取 JSON（支持含额外文本的响应）"""
        # 方法1：正则提取 JSON 数组
        json_match = re.search(r'\[.*\]', response, re.DOTALL)
        if json_match:
            json_str = json_match.group(0)
            try:
                return json.loads(json_str)
            except json.JSONDecodeError as e:
                raise ValueError(f"JSON解析失败：{e}")

        # 方法2：整个响应直接解析
        try:
            return json.loads(response)
        except json.JSONDecodeError:
            raise ValueError("无法从响应中提取JSON")

    def evaluate_plan(self, todo_items: List[TodoItem]) -> dict:
        """评估规划质量"""
        score = 100
        suggestions = []

        if len(todo_items) < 3:
            score -= 20
            suggestions.append("子任务数量过少，可能遗漏重要信息")
        elif len(todo_items) > 5:
            score -= 10
            suggestions.append("子任务数量过多，可能存在冗余")

        for task in todo_items:
            if len(task.query.split()) < 2:
                score -= 10
                suggestions.append(f"任务「{task.title}」的查询过于简单")

        return {"score": score, "suggestions": suggestions}


# 测试 JSON 提取
service = PlanningService()

# 情况1：Agent 响应包含额外文本
response_with_extra_text = '''
好的，我将为您规划以下任务：

[
  {"title": "Datawhale的基本信息", "intent": "了解组织定位", "query": "Datawhale organization introduction 2024"},
  {"title": "Datawhale的主要项目", "intent": "了解核心内容", "query": "Datawhale projects tutorials open source"}
]

这些任务涵盖了主要方面。
'''

tasks = service._extract_tasks(response_with_extra_text)
print(f"情况1（含额外文本）提取结果: {len(tasks)} 个任务")
for t in tasks:
    print(f"  - {t['title']}")

# 情况2：纯 JSON
response_pure_json = '[{"title": "任务A", "intent": "意图A", "query": "query A 2024"}]'
tasks2 = service._extract_tasks(response_pure_json)
print(f"\n情况2（纯JSON）提取结果: {len(tasks2)} 个任务")

# 质量评估
todo_items = [TodoItem(i+1, t['title'], t['intent'], t['query']) for i, t in enumerate(tasks)]
eval_result = service.evaluate_plan(todo_items)
print(f"\n规划质量评估: 得分={eval_result['score']}")

### 14.5.2 总结服务（SummarizationService）

核心职责：格式化搜索结果 → 构建总结 Prompt → 调用 Task Summarizer Agent → 提取来源引用

In [ ]:
from typing import Tuple


class SummarizationService:
    """总结服务"""

    def __init__(self, llm=None, tool_call_listener=None):
        self._llm = llm
        # self._agent = ToolAwareSimpleAgent(name="Task Summarizer", ...)

    def _format_sources(self, search_results: List[dict]) -> str:
        """格式化搜索结果为易读文本"""
        formatted = []
        for idx, result in enumerate(search_results, start=1):
            formatted.append(
                f"[{idx}] {result.get('title', '无标题')}\n"
                f"URL: {result.get('url', '')}\n"
                f"摘要: {result.get('snippet', '')}\n"
            )
        return "\n".join(formatted) # "分隔符".join(列表) 把列表中所有字符串用指定分隔符连接成一个字符串：

    def summarize_task(
        self,
        task: TodoItem,
        search_results: List[dict]
    ) -> Tuple[str, List[str]]:
        """总结任务，返回（总结文本, 来源URL列表）"""
        formatted_sources = self._format_sources(search_results)

        # 实际项目中调用 Agent:
        # prompt = task_summarizer_instructions.format(...)
        # summary = self._agent.run(prompt)

        # 模拟返回
        summary = f"## {task.title}\n\n[模拟总结内容]\n"
        source_urls = [r.get("url", "") for r in search_results]

        return summary, source_urls


# 测试
summarizer = SummarizationService()
mock_search_results = [
    {"title": "Datawhale简介", "url": "https://github.com/datawhalechina", "snippet": "开源AI学习社区..."},
    {"title": "关于Datawhale", "url": "https://datawhale.club/about", "snippet": "成立于2018年..."}
]

sample_task = TodoItem(1, "Datawhale的基本信息", "了解组织定位", "Datawhale organization 2024")
summary, urls = summarizer.summarize_task(sample_task, mock_search_results)

print("格式化后的搜索结果:")
print(summarizer._format_sources(mock_search_results))
print(f"\n提取到的来源URLs: {urls}")

### 14.5.3 报告生成服务（ReportingService）

核心职责：格式化所有子任务总结 → 构建报告 Prompt → 调用 Report Writer Agent → 整理引用

In [ ]:
class ReportingService:
    """报告生成服务"""

    def __init__(self, llm=None, tool_call_listener=None):
        self._llm = llm
        # self._agent = ToolAwareSimpleAgent(name="Report Writer", ...)

    def _format_summaries(
        self,
        task_summaries: List[Tuple[TodoItem, str, List[str]]]
    ) -> str:
        """格式化所有子任务总结（供 Report Writer 使用）"""
        formatted = []
        for idx, (task, summary, source_urls) in enumerate(task_summaries, start=1):
            formatted.append(
                f"## 任务{idx}：{task.title}\n\n"
                f"**意图**：{task.intent}\n\n"
                f"{summary}\n\n"
                f"**来源**：\n"
            )
            for url in source_urls:
                formatted.append(f"- {url}\n")
            formatted.append("\n")
        return "".join(formatted)

    def generate_report(
        self,
        research_topic: str,
        task_summaries: List[Tuple[TodoItem, str, List[str]]]
    ) -> str:
        """生成最终报告（Markdown格式）"""
        formatted_summaries = self._format_summaries(task_summaries)

        # 实际项目中调用 Agent:
        # prompt = report_writer_instructions.format(
        #     research_topic=research_topic,
        #     task_summaries=formatted_summaries,
        # )
        # return self._agent.run(prompt)

        return f"# {research_topic}\n\n[模拟报告内容]\n"


# 测试
reporter = ReportingService()
mock_summaries = [
    (TodoItem(1, "基本信息", "了解定位", "query1"), "总结1内容", ["https://url1.com"]),
    (TodoItem(2, "主要项目", "了解项目", "query2"), "总结2内容", ["https://url2.com", "https://url3.com"]),
]

formatted = reporter._format_summaries(mock_summaries)
print("格式化后的子任务总结（提供给 Report Writer）:")
print(formatted)

### 14.5.4 搜索调度服务（SearchService）

核心职责：根据配置选择搜索引擎 → 执行搜索 → 去重、限制 Token、格式化 → 错误处理与缓存

搜索引擎调度流程（图 14.8）：

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/14-figures/14-8.png" alt="" width="85%"/>
  <p>图 14.8 搜索引擎调度流程</p>
</div>

In [ ]:
import hashlib
import json
import logging
from pathlib import Path
from typing import List

logger = logging.getLogger(__name__)


class SearchService:
    """搜索调度服务（含缓存）"""

    def __init__(self, config=None):
        self.config = config
        # 实际项目中：self.search_tool = SearchTool(backend="hybrid")

        # 缓存目录
        self.cache_dir = Path("./cache/search")
        self.cache_dir.mkdir(parents=True, exist_ok=True)

    def _generate_cache_key(self, query: str, max_results: int) -> str:
        """生成 MD5 缓存键"""
        search_api = getattr(self.config, 'search_api', 'duckduckgo')
        content = f"{query}_{max_results}_{search_api}"
        return hashlib.md5(content.encode()).hexdigest()

    def _deduplicate_sources(self, sources: List[dict]) -> List[dict]:
        seen_urls = set()
        return [
            s for s in sources
            if s.get("url", "") not in seen_urls
            and not seen_urls.add(s.get("url", ""))
        ]

    def _limit_source_tokens(
        self, sources: List[dict], max_tokens_per_source: int = 2000
    ) -> List[dict]:
        max_chars = max_tokens_per_source * 4
        return [
            {**s, "snippet": s.get("snippet", "")[:max_chars] + (
                "..." if len(s.get("snippet", "")) > max_chars else ""
            )}
            for s in sources
        ]

    def search(
        self,
        query: str,
        max_results: int = 5,
        use_cache: bool = True
    ) -> List[dict]:
        """执行搜索（带缓存、去重、Token限制）"""
        cache_key = self._generate_cache_key(query, max_results)
        cache_file = self.cache_dir / f"{cache_key}.json"

        # 尝试从缓存读取
        if use_cache and cache_file.exists():
            with open(cache_file, "r", encoding="utf-8") as f:
                print(f"[缓存命中] {query}")
                return json.load(f)

        # 实际项目中调用 SearchTool
        # raw_response = self.search_tool.run({"input": query, ...})
        # results = raw_response.get("results", [])
        results = [{  # 模拟搜索结果
            "title": f"关于 {query} 的搜索结果",
            "url": f"https://example.com/{hashlib.md5(query.encode()).hexdigest()[:8]}",
            "snippet": f"{query} 的相关内容..." * 10
        }]

        results = self._deduplicate_sources(results)
        results = self._limit_source_tokens(results)

        # 保存到缓存
        if use_cache:
            with open(cache_file, "w", encoding="utf-8") as f:
                json.dump(results, f, ensure_ascii=False, indent=2)

        return results


# 测试
search_svc = SearchService()
results = search_svc.search("Datawhale organization", max_results=3)
print(f"搜索结果数: {len(results)}")
for r in results:
    print(f"  URL: {r['url']}  snippet 长度: {len(r['snippet'])} 字符")

# 第二次搜索应命中缓存
results2 = search_svc.search("Datawhale organization", max_results=3)
print(f"\n再次搜索结果数: {len(results2)} (来自缓存)")

# 清理测试缓存
import shutil
shutil.rmtree("./cache", ignore_errors=True)
print("\n测试缓存已清理")

## 14.6 前端交互设计

### 14.6.1 全屏模态对话框 UI 设计

深度研究助手采用全屏模态对话框 UI，提供沉浸式体验（图 14.9）：

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/14-figures/14-9.png" alt="" width="85%"/>
  <p>图 14.9 全屏模态对话框 UI</p>
</div>

UI 组件：顶部栏（研究主题 + 关闭按钮）、进度区域（进度条 + 状态文本）、内容区域（Markdown 结果）、底部栏（状态信息）。

In [ ]:
# ResearchModal.vue 核心代码（TypeScript + Vue 3）
research_modal_vue = '''
<template>
  <div v-if="isOpen" class="modal-overlay" @click.self="close">
    <div class="modal-container">
      <!-- 顶部栏 -->
      <div class="modal-header">
        <h2>{{ researchTopic }}</h2>
        <button @click="close" class="close-button">×</button>
      </div>

      <!-- 进度区域 -->
      <div class="progress-section">
        <div class="progress-bar">
          <div class="progress-fill" :style="{ width: progressPercentage + \'%\' }"></div>
        </div>
        <div class="progress-text">{{ progressText }}</div>
      </div>

      <!-- 内容区域 -->
      <div class="content-section">
        <div v-if="isLoading" class="loading-spinner">
          <div class="spinner"></div>
          <p>研究中，请稍候...</p>
        </div>
        <div v-else class="markdown-content" v-html="renderedMarkdown"></div>
      </div>

      <!-- 底部栏 -->
      <div class="modal-footer">
        <span class="status-text">{{ statusText }}</span>
      </div>
    </div>
  </div>
</template>

<script setup lang="ts">
import { ref, computed, watch } from \'vue\'
import { marked } from \'marked\'

interface Props {
  isOpen: boolean
  researchTopic: string
}

const props = defineProps<Props>()
const emit = defineEmits<{ close: [] }>()

const isLoading = ref(true)
const progressPercentage = ref(0)
const progressText = ref(\'准备中...\')
const statusText = ref(\'研究中...\')
const markdownContent = ref(\'\')

// 渲染 Markdown
const renderedMarkdown = computed(() => marked(markdownContent.value))

// 关闭模态框
const close = () => emit(\'close\')

// 监听 ESC 键关闭
const handleKeydown = (e: KeyboardEvent) => {
  if (e.key === \'Escape\') close()
}
watch(() => props.isOpen, (isOpen) => {
  isOpen
    ? document.addEventListener(\'keydown\', handleKeydown)
    : document.removeEventListener(\'keydown\', handleKeydown)
})
</script>

<style scoped>
/* 响应式媒体查询 */
@media (max-width: 768px) {
  .modal-container { width: 95vw; height: 95vh; }
}
@media (max-width: 480px) {
  .modal-container { width: 100vw; height: 100vh; border-radius: 0; }
}
</style>
'''

print("ResearchModal.vue 代码:")
print(research_modal_vue)

### 14.6.2 实时进度展示（SSE）

深度研究助手使用 Server-Sent Events（SSE）实现实时进度展示。SSE 流程（图 14.10）：

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/14-figures/14-10.png" alt="" width="85%"/>
  <p>图 14.10 SSE 流程</p>
</div>

1. 客户端发起 POST 请求 → 服务器建立 SSE 连接（`text/event-stream`）
2. 服务器按阶段推送进度（规划 10% → 执行 10%-80% → 报告 90% → 完成 100%）
3. 客户端监听事件，实时更新 UI

In [ ]:
# 后端 FastAPI SSE 端点（Python）
sse_backend_code = '''
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from typing import AsyncGenerator
import json

app = FastAPI()

async def research_stream(topic: str) -> AsyncGenerator[str, None]:
    """SSE 流式研究生成器"""
    try:
        # 1. 规划阶段
        yield f"data: {json.dumps({\n\'type\': \'progress\', \'stage\': \'planning\',
                                    \'percentage\': 10, \'text\': \'正在规划研究任务...\'\n})}\\n\\n"
        todo_items = await planning_service.plan_todo_list(topic)
        yield f"data: {json.dumps({\'type\': \'plan\', \'data\': [item.dict() for item in todo_items]})}\\n\\n"

        # 2. 执行阶段（逐任务）
        task_summaries = []
        for idx, task in enumerate(todo_items, start=1):
            percentage = 10 + (idx / len(todo_items)) * 70
            yield f"data: {json.dumps({\'type\': \'progress\', \'percentage\': percentage,
                                        \'text\': f\'正在研究任务{idx}/{len(todo_items)}：{task.title}\'})}\\n\\n"

            search_results = await search_service.search(task.query)
            summary, source_urls = await summarization_service.summarize_task(task, search_results)
            task_summaries.append((task, summary, source_urls))

            yield f"data: {json.dumps({\'type\': \'task_summary\', \'task_id\': task.id, \'summary\': summary})}\\n\\n"

        # 3. 报告阶段
        yield f"data: {json.dumps({\'type\': \'progress\', \'percentage\': 90, \'text\': \'正在生成最终报告...\'})}\\n\\n"
        report = await reporting_service.generate_report(topic, task_summaries)
        yield f"data: {json.dumps({\'type\': \'report\', \'data\': report})}\\n\\n"

        # 完成
        yield f"data: {json.dumps({\'type\': \'progress\', \'stage\': \'completed\', \'percentage\': 100, \'text\': \'研究完成！\'})}\\n\\n"

    except Exception as e:
        yield f"data: {json.dumps({\'type\': \'error\', \'message\': str(e)})}\\n\\n"

@app.post("/api/research")
async def research(request: ResearchRequest):
    return StreamingResponse(
        research_stream(request.topic),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "Connection": "keep-alive"}
    )
'''

print("FastAPI SSE 端点代码:")
print(sse_backend_code)

In [ ]:
# 前端 EventSource 接收 SSE（TypeScript）
sse_frontend_code = '''
// composables/useResearch.ts
import { ref } from \'vue\'

export function useResearch() {
  const isLoading = ref(false)
  const progressPercentage = ref(0)
  const progressText = ref(\'\')
  const markdownContent = ref(\'\')
  const error = ref<string | null>(null)

  const startResearch = (topic: string) => {
    isLoading.value = true
    error.value = null

    // 建立 SSE 连接
    const eventSource = new EventSource(`/api/research?topic=${encodeURIComponent(topic)}`)

    eventSource.onmessage = (event) => {
      const data = JSON.parse(event.data)

      switch (data.type) {
        case \'progress\':
          progressPercentage.value = data.percentage
          progressText.value = data.text
          break

        case \'plan\':
          console.log(\'规划结果:\', data.data)
          break

        case \'task_summary\':
          // 追加任务总结到 Markdown
          markdownContent.value += `\\n\\n## 任务${data.task_id}\\n\\n${data.summary}`
          break

        case \'report\':
          markdownContent.value = data.data
          break

        case \'error\':
          error.value = data.message
          eventSource.close()
          isLoading.value = false
          break

        case \'completed\':
          eventSource.close()
          isLoading.value = false
          break
      }
    }

    eventSource.onerror = (err) => {
      error.value = \'连接失败，请重试\'
      eventSource.close()
      isLoading.value = false
    }
  }

  return { isLoading, progressPercentage, progressText, markdownContent, error, startResearch }
}
'''

print("前端 useResearch composable:")
print(sse_frontend_code)

### 14.6.3 研究结果可视化

研究结果以 Markdown 格式展示，使用 `marked` 库将 Markdown 转换为 HTML，并添加自定义样式。

In [ ]:
# Markdown 渲染与来源引用处理（TypeScript）
markdown_rendering_code = '''
import { marked } from \'marked\'

// 配置 marked（支持 GFM 和换行）
marked.setOptions({
  breaks: true,  // 支持单行换行
  gfm: true,     // 支持 GitHub Flavored Markdown
})

// 渲染 Markdown 为 HTML
const renderedHtml = marked(markdownContent.value)
'''

# 最终报告的来源引用示例
example_report_references = """
## 参考文献

### 任务1：Datawhale的基本信息
- [Datawhale GitHub](https://github.com/datawhalechina)
- [Datawhale 官网](https://datawhale.club)

### 任务2：Datawhale的主要项目
- [Hello-Agents 教程](https://github.com/datawhalechina/Hello-Agents)
- [Joyful-Pandas 教程](https://github.com/datawhalechina/joyful-pandas)
"""

print("Markdown 渲染代码:")
print(markdown_rendering_code)
print("\n报告来源引用格式示例:")
print(example_report_references)

## 14.7 本章小结

在本章中，我们从零开始构建了一个完整的自动化深度研究智能体系统。核心要点回顾：

**（1）TODO 驱动的研究范式**

三阶段流程：
- **规划阶段**：将研究主题分解为 3-5 个子任务（title、intent、query）
- **执行阶段**：对每个子任务执行搜索和总结，保留来源引用
- **报告阶段**：整合所有总结，生成结构化的最终报告

**（2）三 Agent 协作系统**

| Agent | 职责 | 关键设计 |
|-------|------|----------|
| TODO Planner | 分解研究主题 | JSON 输出、逻辑关系约束 |
| Task Summarizer | 总结搜索结果 | 来源引用、关键数据保留 |
| Report Writer | 生成最终报告 | 结构化输出、消除冗余 |

**（3）ToolAwareSimpleAgent 的设计**

扩展 `SimpleAgent`，增加 `tool_call_listener` 回调，实现工具调用监听和实时进度推送。

**（4）工具系统集成**
- **SearchTool**：扩展支持 Tavily、DuckDuckGo、Perplexity、SearXNG 等多搜索引擎
- **NoteTool**：持久化研究进度，支持恢复和审计
- **ToolRegistry**：统一管理所有工具

**（5）四个核心服务**
- `PlanningService`：JSON 解析与验证，规划质量评估
- `SummarizationService`：格式化搜索结果，提取来源
- `ReportingService`：整合总结，生成报告
- `SearchService`：搜索调度、去重、Token 限制、缓存

**（6）前端交互设计**
- 全屏模态对话框：沉浸式体验
- SSE 实时进度：实时展示研究阶段和进度百分比
- Markdown 可视化：美观的报告展示，包含来源引用

---

这些知识不仅适用于深度研究助手，也可以应用到其他 AI 应用中。在下一章中，我们将构建一个与游戏引擎结合的多 Agent 系统——赛博小镇，探索 Agent 之间的复杂交互和协作模式。敬请期待！